# Pancancer enrichment analysis step 5: Find enriched pathways using GSEApy

## Setup

In [ ]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime
import glob

In [ ]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP4_DIR = "step4_outputs"

STEP5_DIR = "step5_outputs"
STEP5_FILE = f"enrichment_gseapy_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
STEP5_FILE_PATH = os.path.join(STEP5_DIR, STEP5_FILE)

GSEAPY_DIR_PATH = os.path.join(STEP5_DIR, "gseapy")

if not os.path.isdir(STEP5_DIR):
    os.mkdir(STEP5_DIR)

## Prepare data

In [ ]:
# Read in the files from step 4
step4_files = glob.glob(f"{STEP4_DIR}{os.sep}*")
prot_dict = {}
for file in step4_files:
    name = file.split(os.sep)[1].split("_")[0]
    prot_dict[name] = pd.read_csv(file, sep='\t', index_col=0)

In [ ]:
prot_dict.keys()

## Run enrichment analysis

In [ ]:
# For each cancer, find enriched pathways based on p value for differential expression   
all_enrichments = pd.DataFrame()

for cancer_type in prot_dict.keys():
    
    ranked_data = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "one_minus_adj_p"]].\
        set_index("protein_str").\
        rename(columns={"one_minus_adj_p": f"{cancer_type}_one_minus_adj_p"})
    
    gs_res = gp.gsea(
        data=ranked_data,
        gene_sets='Reactome_2016',
        cls= './data/P53.cls', # TODO: Fix this
        permutation_type='phenotype',
        permutation_num=10,
        min_size=15,
        max_size=500, 
        outdir=GSEAPY_DIR_PATH,
        no_plot=True,
        method="abs_signal_to_noise",
        processes=4,
        seed=0)
    
    cancer_enriched = gs_res.res2d.assign(cancer_type=cancer_type)
    all_enrichments = all_enrichments.append(cancer_enriched)

In [ ]:
all_enrichments.head()

## Summarize enrichment results from all cancers

In [ ]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that pathway is enriched
# - The average of the adjusted p-values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["stId", "name"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["stId"].apply(
    lambda x: all_enrichments[all_enrichments["stId"] == x].shape[0])

avg_fdr = enrichment_summary["stId"].apply(
    lambda x: all_enrichments.loc[all_enrichments["stId"] == x, "entities_fdr"].mean())

avg_overlap_props = enrichment_summary["stId"].apply(
    lambda x: all_enrichments.loc[all_enrichments["stId"] == x, "entities_ratio"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    avg_fdr=avg_fdr,
    avg_overlap_prop=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(
    by=["times_enriched", "avg_fdr"],
    ascending=[False, True])

enrichment_summary = enrichment_summary.reset_index(drop=True)

In [ ]:
enrichment_summary.head()

## Visualize pathways with expression change

In [ ]:
# Submit the differential expression data for each cancer type to the analysis service, and get the tokens
# The data we'd submitted previously were the p values for the differential expression, but that's not what
# we want to visualize; we want to see whether the expression was up or down.
diff_expr_tokens = {}

for cancer_type in data["cancer_type"].unique():

    # Select data for that cancer type
    omics = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "simplified_change"]].\
        set_index("protein_str").\
        rename(columns={"simplified_change": f"{cancer_type}_simp_change"})

    analysis_token, unneeded_df = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=omics, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    diff_expr_tokens[cancer_type] = analysis_token

In [ ]:
# Select pathways to get URLs for
to_viz = enrichment_summary[["stId", "name"]].iloc[0:10, :]

id_list = []
name_list = []
cancer_type_list = []
expr_list = []
url_list = []

for idx in to_viz.index:
    pathway_id = to_viz.loc[idx, "stId"]
    name = to_viz.loc[idx, "name"]
    
    for cancer_type in data["cancer_type"].unique():
         
        # Get the URL
        expr_vals, url = ut.reactome_pathway_overlay(
            pathway=pathway_id,
            analysis_token=diff_expr_tokens[cancer_type],
            open_browser=False)
        
        id_list.append(pathway_id)
        name_list.append(name)
        cancer_type_list.append(cancer_type)
        expr_list.append(expr_vals[0])
        url_list.append(url)

urls_df = pd.DataFrame(
    {
        "pathway_id": id_list,
        "pathway_name": name_list,
        "cancer_type": cancer_type_list,
        "mean_expr": expr_list,
        "url": url_list
    })

In [ ]:
# Print the dataframe in such a way that the links are clickable
urls_df.style.format('<a href="{0}">{0}</a>', subset="url")

In [ ]:
# import webbrowser
# for url in urls_df["url"][0:8]:
#     webbrowser.open(url)

In [ ]:
# Save the results
urls_df.to_csv(STEP5_FILE_PATH, sep='\t')